# 00b -- Prepare Raw Datasets into the Unified Schema

Run this **after** `00_colab_setup_and_drive_mount.ipynb` and **before**
`01_stage1_train_image_datasets.ipynb`.

`00_colab_setup_and_drive_mount.ipynb` only checks that each raw dataset
folder *exists* under `data.root_drive`. Training itself
(`01_stage1_...ipynb`) does **not** read those raw folders directly -- it
reads one unified `labels.csv` plus `images/` and `masks/` folders. This
notebook converts each raw dataset's own layout into that unified schema.

Covers: CASIA2.0, CoMoFoD, Coverage, IMD2020, FaceForensics++ (c23).
DFDC is handled separately by `02_stage2_dfdc_frame_extraction_and_train.ipynb`
(run on Kaggle, not here).

**Expected starting point on Drive** (from manually uploading/extracting
your zips -- see the project README / setup guide for exact steps):
```
ForensicAI_Lab/data/
    casia2/...        (raw CASIA2.0 contents)
    comofod/...       (raw CoMoFoD contents)
    coverage/...      (raw Coverage contents)
    imd2020/...       (raw IMD2020 contents)
    ffpp_videos/...   (raw FaceForensics++ c23 contents)
```

**Important**: the converters below encode each dataset's *typical/documented*
folder layout, which can vary slightly between re-uploads and mirrors.
**Always run the inspection cell first** and compare its output against the
comment at the top of each converter cell before trusting the result. If a
converter reports 0 matched rows for a dataset you know has files, the
layout differs from what's assumed here -- share the inspection output and
adjust the glob patterns (or ask for help adjusting them) before proceeding.

In [ ]:
import sys, os, subprocess
PROJECT_ROOT = "/content/ForensicAI_Lab"  # adjust if you cloned/uploaded elsewhere
if not os.path.exists(PROJECT_ROOT):
    PROJECT_ROOT = os.getcwd()  # fallback: running locally from the project root
sys.path.insert(0, PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, is_colab

if is_colab():
    mount_drive()

# This notebook only prepares data -- no torch needed, so it's a much
# lighter install than requirements-train.txt.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "pandas", "opencv-python-headless", "PyYAML", "pydantic"])

In [ ]:
from pathlib import Path
from config.loader import get_settings, resolve_dataset_root

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
print("Dataset root:", dataset_root)

## 1. Inspect what's actually on disk (run this first, every time)

In [ ]:
def inspect(folder: Path, max_items: int = 15) -> None:
    if not folder.exists():
        print(f"  [MISSING] {folder}")
        return
    files = [f for f in folder.rglob("*") if f.is_file()]
    print(f"  [OK] {folder} -- {len(files)} files")
    exts = {}
    for f in files:
        exts[f.suffix.lower()] = exts.get(f.suffix.lower(), 0) + 1
    print("    extensions:", exts)
    print("    sample paths:")
    for f in files[:max_items]:
        print("     ", f.relative_to(folder))

for entry in settings.data.datasets_registry:
    if entry.local_subdir == "dfdc_subset":
        continue  # handled on Kaggle, not here
    print(f"\n=== {entry.name} ({entry.local_subdir}) ===")
    inspect(dataset_root / entry.local_subdir)

## 2. Convert each dataset into unified rows

Each converter appends `{filename, label, type}` dicts to `rows` and
copies the underlying image (and mask, if available) into
`dataset_root/images/` and `dataset_root/masks/`, prefixed by dataset name
so filenames can't collide across sources.

In [ ]:
import shutil, cv2

images_out = dataset_root / settings.data.images_dir
masks_out = dataset_root / settings.data.masks_dir
images_out.mkdir(parents=True, exist_ok=True)
masks_out.mkdir(parents=True, exist_ok=True)

rows = []


def add_row(src_image, label, manip_type, src_mask=None, prefix=""):
    dst_name = f"{prefix}{src_image.name}"
    shutil.copy(src_image, images_out / dst_name)
    if src_mask is not None and src_mask.exists():
        shutil.copy(src_mask, masks_out / dst_name)
    rows.append({"filename": dst_name, "label": label, "type": manip_type})

In [ ]:
# CASIA2.0 -- standard release layout: casia2/Au/*  (authentic),
# casia2/Tp/*  (tampered). Tampered filenames encode the forgery type as
# the 2nd underscore-separated token: 'D' = different source (splicing),
# 'S' = same source (copy-move), e.g. Tp_D_CND_M_N_ani00018_sec00096_00138.jpg
# No official ground-truth masks ship with CASIA2.0 -- masks skipped here.
casia_dir = dataset_root / "casia2"
au_dir, tp_dir = casia_dir / "Au", casia_dir / "Tp"
_before = len(rows)

for img in (au_dir.glob("*.*") if au_dir.exists() else []):
    add_row(img, "real", "none", prefix="casia2_")

for img in (tp_dir.glob("*.*") if tp_dir.exists() else []):
    parts = img.stem.split("_")
    manip = "splicing" if len(parts) > 1 and parts[1].upper() == "D" else "copy_move"
    add_row(img, "fake", manip, prefix="casia2_")

print(f"CASIA2.0: added {len(rows) - _before} rows")

In [ ]:
# CoMoFoD -- flat folder, samples share a numeric prefix:
#   <id>_F.<ext> = forged (copy-move) image
#   <id>_O.<ext> = original/authentic image
#   <id>_B.<ext> = binary ground-truth mask for the forged image
comofod_dir = dataset_root / "comofod"
_before = len(rows)

if comofod_dir.exists():
    for forged in comofod_dir.glob("*_F.*"):
        sample_id = forged.stem.rsplit("_", 1)[0]
        mask_candidates = list(comofod_dir.glob(f"{sample_id}_B.*"))
        mask = mask_candidates[0] if mask_candidates else None
        add_row(forged, "fake", "copy_move", src_mask=mask, prefix="comofod_")

    for original in comofod_dir.glob("*_O.*"):
        add_row(original, "real", "none", prefix="comofod_")

print(f"CoMoFoD: added {len(rows) - _before} rows")

In [ ]:
# Coverage -- coverage/image/ holds <id>.tif (original) and <id>t.tif
# (tampered); coverage/mask/ holds the matching ground-truth mask.
coverage_dir = dataset_root / "coverage"
image_dir, mask_dir = coverage_dir / "image", coverage_dir / "mask"
_before = len(rows)

if image_dir.exists():
    for img in image_dir.glob("*t.*"):
        sample_id = img.stem[:-1]
        mask_candidates = list(mask_dir.glob(f"{sample_id}*.*")) if mask_dir.exists() else []
        mask = mask_candidates[0] if mask_candidates else None
        add_row(img, "fake", "copy_move", src_mask=mask, prefix="coverage_")

    for img in image_dir.glob("*.*"):
        if img.stem.endswith("t"):
            continue
        add_row(img, "real", "none", prefix="coverage_")

print(f"Coverage: added {len(rows) - _before} rows")

In [ ]:
# IMD2020 -- layout varies more across mirrors/releases than the other
# three, so this is a heuristic, not a documented spec: walk the whole
# tree, treat any filename containing "mask" as a ground-truth mask,
# anything containing "orig" as authentic, and any other image as the
# manipulated counterpart (type defaults to "splicing" since IMD2020 is
# real-world manipulations without a reliable per-sample type label).
# ** Verify this one especially carefully against the inspection output. **
imd_dir = dataset_root / "imd2020"
_before = len(rows)

if imd_dir.exists():
    all_files = [f for f in imd_dir.rglob("*") if f.is_file()]
    masks = {f.stem.replace("_mask", "").replace("mask", ""): f
             for f in all_files if "mask" in f.name.lower()}

    for f in all_files:
        name_lower = f.name.lower()
        if "mask" in name_lower:
            continue
        if "orig" in name_lower:
            add_row(f, "real", "none", prefix="imd2020_")
        else:
            add_row(f, "fake", "splicing", src_mask=masks.get(f.stem), prefix="imd2020_")

print(f"IMD2020: added {len(rows) - _before} rows")

In [ ]:
# FaceForensics++ (c23, video) -- official layout:
#   ffpp_videos/original_sequences/youtube/c23/videos/*.mp4        (real)
#   ffpp_videos/manipulated_sequences/<Method>/c23/videos/*.mp4    (fake)
# where <Method> is Deepfakes/Face2Face/FaceSwap/NeuralTextures/etc.
# Kaggle re-uploads sometimes flatten or rename this layout -- verify
# against the inspection output above before trusting the result. Only a
# few frames per video are sampled (not every frame) to avoid exploding
# the combined image count.
from services.video.extractor import extract_frames, get_video_fps, get_frame_count
from services.video.sampler import sample_frame_indices

ffpp_dir = dataset_root / "ffpp_videos"
_before = len(rows)


def _extract_ffpp_frames(video_path, label, manip_type, max_frames=5):
    fps = get_video_fps(str(video_path))
    total = get_frame_count(str(video_path))
    indices = set(sample_frame_indices(total, fps, settings.video.sample_fps, max_frames=max_frames))
    for idx, frame in extract_frames(str(video_path)):
        if idx not in indices:
            continue
        frame_name = f"ffpp_{manip_type}_{video_path.stem}_frame{idx}.jpg"
        cv2.imwrite(str(images_out / frame_name), frame)
        rows.append({"filename": frame_name, "label": label, "type": manip_type})


if ffpp_dir.exists():
    real_dir = ffpp_dir / "original_sequences" / "youtube" / "c23" / "videos"
    for video_path in (real_dir.glob("*.mp4") if real_dir.exists() else []):
        _extract_ffpp_frames(video_path, "real", "none")

    manip_root = ffpp_dir / "manipulated_sequences"
    if manip_root.exists():
        for method_dir in manip_root.iterdir():
            video_dir = method_dir / "c23" / "videos"
            for video_path in (video_dir.glob("*.mp4") if video_dir.exists() else []):
                _extract_ffpp_frames(video_path, "fake", "gan")

print(f"FaceForensics++ c23: added {len(rows) - _before} rows")

## 3. Write the combined `labels.csv`

Appends to (and de-duplicates against) whatever's already in
`dataset_root/labels.csv` -- safe to re-run after fixing a converter above.

In [ ]:
import pandas as pd

new_df = pd.DataFrame(rows, columns=["filename", "label", "type"])
labels_path = dataset_root / settings.data.labels_csv

if labels_path.exists():
    existing = pd.read_csv(labels_path)
    combined = pd.concat([existing, new_df], ignore_index=True).drop_duplicates(subset="filename")
else:
    combined = new_df

combined.to_csv(labels_path, index=False)
print(f"Wrote {len(combined)} total rows to {labels_path}")
print(combined.groupby(["label", "type"]).size())